# 中证500超额收益预测
使用相同五个特征比较 OLS、LASSO、随机森林和神经网络。数据与收益定义及限制见研究结果说明。下方依次运行将重新估计模型并覆盖结果目录对应预测文件。

In [ ]:
from pathlib import Path
import os, sys, json, hashlib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import make_pipeline
import joblib
P = Path.cwd()
if not (P / '结果').exists(): P = P / '研究最终交付'
DATA = P.parent / 'method_results' / 'csi500_stock_month.csv'
FEATURES = ['momentum','reversal','size','turnover','volatility']
SEEDS = [3080,3081,3082,3083,3084]
assert DATA.exists(), f'找不到输入数据: {DATA}'


## 数据处理、模型估计与样本外预测
参数已根据2021–2022年验证期选定；每年仅使用预测形成日前已结束的收益标签。

In [ ]:
def load_panel():
 p=pd.read_csv(DATA,parse_dates=['formation_date','entry_date','exit_date'])
 assert not p.duplicated(['stock','formation_date']).any()
 for c in FEATURES:
  assert p.groupby('holding_month')[c].count().gt(1).all()
  p[c]=p.groupby('holding_month')[c].transform(lambda x:(2*(x.rank()-1)/(x.count()-1)-1).fillna(0))
 assert np.isfinite(p[FEATURES]).all().all()
 return p

def split(p,year):
 v=p[p.holding_month.str.startswith(str(year))].copy()
 t=p[p.exit_date.le(v.formation_date.min()) & p.excess_return.notna()]
 assert t.exit_date.max()<=v.formation_date.min()
 return t,v

def nn_path(setting,seed,x,y,z,epochs,save=None):
 """No validation target is passed: returns fixed checkpoints for external selection."""
 scale=StandardScaler();x=scale.fit_transform(x).astype('float32');z=scale.transform(z).astype('float32');y=np.asarray(y,dtype='float32')
 pred={}
 if setting['engine']=='sklearn':
  # Persist RNG state across partial_fit calls, avoiding RNG reset at every epoch.
  net=MLPRegressor(hidden_layer_sizes=tuple(setting['layers']),alpha=setting['alpha'],learning_rate_init=setting['lr'],batch_size=setting['batch'],random_state=np.random.RandomState(seed),early_stopping=False)
  for epoch in range(1,max(epochs)+1):
   net.partial_fit(x,y)
   if epoch in epochs:pred[epoch]=net.predict(z)
  if save:joblib.dump({'model':net,'scaler':scale,'features':FEATURES},save.with_suffix('.joblib'))
 else:
  import sys
  if Path('/tmp/eco3080-torch').exists():sys.path.insert(0,'/tmp/eco3080-torch')
  import torch
  torch.set_num_threads(2);torch.manual_seed(seed)
  from torch import nn
  layers=[];previous=len(FEATURES)
  for width in setting['layers']:
   linear=nn.Linear(previous,width);nn.init.kaiming_normal_(linear.weight,nonlinearity='relu');nn.init.zeros_(linear.bias)
   layers += [linear,nn.ReLU(),nn.BatchNorm1d(width,eps=.001,momentum=.01)];previous=width
  output=nn.Linear(previous,1);nn.init.zeros_(output.weight);nn.init.constant_(output.bias,float(y.mean()));layers.append(output)
  net=nn.Sequential(*layers);opt=torch.optim.Adam(net.parameters(),lr=setting['lr'],eps=1e-7)
  xt=torch.from_numpy(x);yt=torch.from_numpy(y);zt=torch.from_numpy(z)
  weights=[layer.weight for layer in net if isinstance(layer,nn.Linear)]
  rng=np.random.default_rng(seed);batch=setting['batch']
  for epoch in range(1,max(epochs)+1):
   net.train();order=rng.permutation(len(y))
   for start in range(0,len(y),batch):
    ix=order[start:start+batch];opt.zero_grad(set_to_none=True)
    estimate=net(xt[ix]).squeeze(1)
    loss=(estimate-yt[ix]).square().mean()+setting['l1']*sum(w.abs().sum() for w in weights)
    loss.backward();opt.step()
   assert np.isfinite(float(loss.detach()))
   if epoch in epochs:
    net.eval()
    with torch.no_grad():pred[epoch]=net(zt).squeeze(1).numpy().copy()
   if epoch in [20,60,100]:print('NN checkpoint',seed,epoch,flush=True)
  if save:
   torch.save({'state_dict':net.state_dict(),'setting':setting,'features':FEATURES},save.with_suffix('.pt'))
   joblib.dump(scale,save.with_suffix('.scaler.joblib'))
 for value in pred.values():assert np.isfinite(value).all()
 return pred

def evaluate():
 settings=json.loads((P/'结果/selected_settings.json').read_text());assert hashlib.sha256(DATA.read_bytes()).hexdigest()==settings['metadata']['data_sha256']
 p=load_panel();parts=[];availability=[];seedparts=[]
 for year in [2023,2024,2025,2026]:
  t,v=split(p,year)
  if year==2026:v=v[v.holding_month.eq('2026-01')].copy()
  availability.append(dict(year=year,cutoff=str(v.formation_date.min()),last_label_exit=str(t.exit_date.max()),rows=len(t)))
  x,y,z=t[FEATURES].to_numpy(),t.excess_return.to_numpy(),v[FEATURES].to_numpy()
  for name in ['OLS','LASSO','RF']:
   if name=='OLS':model=LinearRegression()
   elif name=='LASSO':
    from sklearn.pipeline import make_pipeline
    model=make_pipeline(StandardScaler(),Lasso(**settings[name],max_iter=10000))
   else:model=RandomForestRegressor(**settings[name],random_state=3080,n_jobs=6)
   model.fit(x,y);v[name]=model.predict(z)
   if year==2026:joblib.dump(model,P/'结果'/(name+'_final.joblib'))
  ns=dict(settings['MLP']);epoch=ns.pop('epochs');preds=[]
  for seed in SEEDS:
   pred=nn_path(ns,seed,x,y,z,[epoch],P/'结果'/f'MLP_final_seed{seed}' if year==2026 else None)[epoch];preds.append(pred)
   a=v[['stock','holding_month','excess_return']].copy();a['seed']=seed;a['prediction']=pred;seedparts.append(a)
  v['MLP']=np.mean(preds,axis=0);v['MLP_seed_std']=np.std(preds,axis=0,ddof=1);parts.append(v)
  print('Forecasts complete',year,flush=True)
 allf=pd.concat(parts);test=allf[allf.holding_month.between('2023-01','2025-12')];app=allf[allf.holding_month.eq('2026-01')]
 assert len(test)==18000 and len(test.dropna(subset=['excess_return']))==17998
 test.to_csv(P/'结果/all_test_forecasts.csv',index=False);test.dropna(subset=['excess_return']).to_csv(P/'结果/test_forecasts.csv',index=False);app.to_csv(P/'结果/application_2026_01_forecasts.csv',index=False)
 pd.concat(seedparts).to_csv(P/'结果/seed_forecasts.csv',index=False);pd.DataFrame(availability).to_csv(P/'结果/training_availability.csv',index=False)
 for frame,file in [(test,'prediction_metrics.csv'),(app,'application_2026_01_prediction_metrics.csv')]:
  q=frame.dropna(subset=['excess_return']);out=[]
  for m in ['OLS','LASSO','RF','MLP']:
   err=q.excess_return-q[m];out.append(dict(model=m,**{'OOS R2 (%)':100*(1-(err**2).sum()/(q.excess_return**2).sum()),'RMSE (%)':100*np.sqrt((err**2).mean()),'MAE (%)':100*err.abs().mean()}))
  pd.DataFrame(out).sort_values('OOS R2 (%)',ascending=False).to_csv(P/'结果'/file,index=False)
 # Main OLS comparisons, same monthly HAC lag as baseline.
 import statsmodels.api as sm
 q=test.dropna(subset=['excess_return']);out=[]
 for m in ['LASSO','RF','MLP']:
  d=((q.excess_return-q.OLS)**2-(q.excess_return-q[m])**2).groupby(q.holding_month).mean()
  fit=sm.OLS(d,np.ones((len(d),1))).fit(cov_type='HAC',cov_kwds={'maxlags':3})
  out.append({'model vs OLS':m,'mean monthly MSE improvement':d.mean(),'NW t-stat':fit.tvalues.iloc[0],'NW p-value':fit.pvalues.iloc[0],'months':len(d)})
 pd.DataFrame(out).to_csv(P/'结果/newey_west_tests.csv',index=False)
 print(pd.read_csv(P/'结果/prediction_metrics.csv').to_string(index=False),flush=True)

evaluate()

## 预测精度及统计比较

In [ ]:
from IPython.display import display
display(pd.read_csv(P/'结果/prediction_metrics.csv'))
display(pd.read_csv(P/'结果/newey_west_tests.csv'))

## 投资组合结果
根据形成时预测对每月股票分组；下表使用已保存的组合结果。

In [ ]:
port=pd.read_csv(P/'结果/portfolio_comparison.csv')
display(port[(port.weight=='EW') & (port.cost_bps_per_side==0)])
display(pd.read_csv(P/'结果/application_2026_01_prediction_metrics.csv'))